# Source-Grounded Constrained Decoding — Colab runner

Run this on T4 if you don't want it to take 10000000000 years XD


## 1 · Mount Drive and enter the project

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/constrained-decoder"

from google.colab import drive
drive.mount("/content/drive")

%cd $PROJECT_PATH
!ls -1 src scripts | head

## 2 · Install the extras Colab doesn't have natively

In [ ]:
!pip install -q -r requirements-colab.txt
!pip install -q scipy
!python -m spacy download en_core_web_sm -q

## 3 · Authenticate with Hugging Face

Put your secrets token in the sidebar or just paste into login()

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Loaded HF token from Colab secrets.")
except Exception as exc:
    print(f"Colab secret not available ({exc}); falling back to interactive login.")
    from huggingface_hub import login
    login()

## 4 · Experiment configuration

Default model = Llama-3.2-1B-Instruct; swap to a non-gated open-weights model below if you'd rather skip the licence step.

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
# Non-gated alternatives (uncomment one):
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

DATASET = "EdinburghNLP/xsum"
SPLIT = "test"
N_EXAMPLES = 200          # 200 for the headline run; drop to 50 for a quick smoke test
MAX_NEW_TOKENS = 96       # ↑ from 64 — single-sentence summaries were being truncated
SOFT_PENALTY = 5.0        # used only when SOFT_PENALTY_SWEEP is None
SEED = 42

SOFT_PENALTY_SWEEP = (2, 5, 10, 20, 40)

# fixes (still in force)
STRIP_PREAMBLE = True                 # post-process "Here is a concise summary…" preambles
MAX_CONSECUTIVE_CONSTRAINED = 6       # watchdog: deactivate detector after this many constrained steps in a row
OUTPUT_TAG = "v3_sweep" if SOFT_PENALTY_SWEEP else "v3"

# Stricter instruction: explicitly forbids the preamble the v1 prompt produced.
INSTRUCTION = (
    "Summarise the following article in a single concise sentence. "
    "Only state facts that appear in the article. "
    "Do not include any preamble, header, bullet points, or explanation — "
    "output only the summary sentence itself."
)

OUTPUT_JSON = f"results/raw/xsum_{N_EXAMPLES}_{OUTPUT_TAG}.json"
FIGURES_DIR = f"results/figures_{OUTPUT_TAG}"

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 5 · Pre-cache the dataset

Avoids a redownload on every runtime restart.

In [ ]:
!python scripts/download_data.py --dataset $DATASET --splits test

## 6 · Run the experiment (watchdog detector + preamble-stripped post-processing)

In [ ]:
import sys, json, time, random, re, logging
from pathlib import Path
if "." not in sys.path: sys.path.insert(0, ".")

from scripts.run_experiment import (
    ExperimentConfig,
    load_model_and_tokenizer, load_dataset_examples, build_prompt,
    aggregate_metrics, _flatten_metrics,
)
from src.entity_extractor import extract_facts
from src.evaluate_faithfulness import evaluate_faithfulness
from src.evaluate_quality import evaluate_quality
from src.generate import generate_summaries
from src.detector import HeuristicFactualSpanDetector

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

class WatchdogDetector(HeuristicFactualSpanDetector):
    """Wraps the heuristic; force-deactivates after a constrained-streak cap.

    The streak resets when the most recent emitted character is a clause
    terminator (so a long entity span like ``Jessica Ennis-Hill`` still
    completes), or when the underlying heuristic deactivates on its own.
    """

    def __init__(self, *args, max_consecutive=MAX_CONSECUTIVE_CONSTRAINED, **kwargs):
        super().__init__(*args, **kwargs)
        self.max_consecutive = max_consecutive
        self._streak = 0

    def reset(self):
        super().reset()
        self._streak = 0

    def is_active_from_text(self, text: str) -> bool:
        base = super().is_active_from_text(text)
        if not base:
            self._streak = 0
            return False
        # Reset streak at clause boundaries even if the heuristic still fires.
        if text and text[-1] in self.terminator_chars:
            self._streak = 0
        if self._streak >= self.max_consecutive:
            # Force-deactivate; do not increment streak.
            return False
        self._streak += 1
        return True

# strip chatty preambles from generated summaries 
_PREAMBLE_PATTERNS = [
    re.compile(r"^\s*here(?:'s| is)[^\n:]*(?:summary|sentence|key points|article)[^\n:]*:\s*\n+", re.I),
    re.compile(r"^\s*(?:sure|certainly|of course)[^\n]*\n+", re.I),
    re.compile(r"^\s*summary\s*:\s*\n+", re.I),
    re.compile(r"^\s*\*+\s*", re.M),  # bullet artefacts at line starts
]

def strip_preamble(text: str) -> str:
    out = text
    for pat in _PREAMBLE_PATTERNS[:3]:  # only the leading-preamble ones, not the bullet stripper
        out = pat.sub("", out, count=1)
    return out.strip()

# run
config = ExperimentConfig(
    model=MODEL_NAME, dataset=DATASET, split=SPLIT, n=N_EXAMPLES, seed=SEED,
    max_new_tokens=MAX_NEW_TOKENS, soft_penalty=SOFT_PENALTY,
    dtype="float16", device=None,
    output=OUTPUT_JSON, instruction=INSTRUCTION,
    soft_penalty_sweep=SOFT_PENALTY_SWEEP,
)
CONDITIONS_RUN = config.conditions  # tuple of condition names (sweep-aware)
print("Conditions for this run:", CONDITIONS_RUN)

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

tokenizer, model = load_model_and_tokenizer(MODEL_NAME, dtype="float16", device=None)
examples = load_dataset_examples(DATASET, SPLIT, N_EXAMPLES, SEED)

try:
    from tqdm.auto import tqdm
    iterator = tqdm(examples, desc="examples")
except ImportError:
    iterator = examples

per_example = []
start = time.time()
for example in iterator:
    try:
        article, reference = example["article"], example["reference"]
        prompt = build_prompt(article, tokenizer, INSTRUCTION)
        source_facts = extract_facts(article, tokenizer=tokenizer)

        # Fresh watchdog per example — its internal streak counter resets per generation,
        # but a new instance also guarantees no leakage between examples.
        detector = WatchdogDetector()

        summaries = generate_summaries(
            article, prompt, model, tokenizer,
            max_new_tokens=MAX_NEW_TOKENS,
            soft_penalty=SOFT_PENALTY,
            soft_penalty_sweep=SOFT_PENALTY_SWEEP,
            source_facts=source_facts,
            detector=detector,
        )

        cond_records = {}
        for name, summary in summaries.items():
            raw_text = summary.text
            cleaned = strip_preamble(raw_text) if STRIP_PREAMBLE else raw_text
            faith = evaluate_faithfulness(cleaned, article, source_facts=source_facts)
            quality = evaluate_quality(cleaned, reference=reference, tokenizer=tokenizer)
            rec = _flatten_metrics(
                condition_name=name, text=cleaned,
                new_token_ids=summary.new_token_ids,
                fraction_constrained=summary.fraction_constrained,
                faith=faith, quality=quality,
            )
            rec["text_raw"] = raw_text  # keep the unstripped string for diagnostics
            cond_records[name] = rec

        per_example.append({
            "idx": example["idx"], "article": article, "reference": reference,
            "prompt": prompt, "conditions": cond_records,
        })
    except Exception as exc:
        logging.exception("Example idx=%s failed: %s", example.get("idx"), exc)

elapsed = time.time() - start
aggregate = aggregate_metrics(per_example, conditions=CONDITIONS_RUN)

result = {
    "config": {**config.as_dict(),
               "strip_preamble": STRIP_PREAMBLE,
               "max_consecutive_constrained": MAX_CONSECUTIVE_CONSTRAINED,
               "version": "v3"},
    "elapsed_seconds": elapsed,
    "n_examples_processed": len(per_example),
    "conditions": list(CONDITIONS_RUN),
    "per_example": per_example,
    "aggregate": aggregate,
}
Path(OUTPUT_JSON).parent.mkdir(parents=True, exist_ok=True)
Path(OUTPUT_JSON).write_text(json.dumps(result, indent=2))
print(f"Wrote {OUTPUT_JSON} — processed {len(per_example)} examples in {elapsed:.1f}s")


## 7 · Aggregate metrics

In [ ]:
import json
import pandas as pd

with open(OUTPUT_JSON) as f:
    result = json.load(f)

rows = []
for cond, metrics in result["aggregate"].items():
    rows.append({
        "condition": cond,
        "entity_precision": metrics["entity_precision"]["mean"],
        "entity_recall": metrics["entity_recall"]["mean"],
        "hallucinated_per_summary": metrics["hallucinated_entity_count"]["mean"],
        "number_accuracy": metrics["number_accuracy"]["mean"],
        "rouge1": metrics["rouge1"]["mean"],
        "rouge2": metrics["rouge2"]["mean"],
        "rougeL": metrics["rougeL"]["mean"],
        "length_tokens": metrics["length_tokens"]["mean"],
        "frac_constrained": metrics["fraction_constrained"]["mean"],
    })
df = pd.DataFrame(rows).set_index("condition")
df.round(3)

## 7a · Diagnostics: preamble, truncation, runaway detector

In [ ]:
import json
import re
from collections import Counter
with open(OUTPUT_JSON) as f:
    result = json.load(f)
per_ex = result["per_example"]
CONDITIONS_RUN = tuple(result["conditions"])
N = len(per_ex)
cap = MAX_NEW_TOKENS

preamble_re = re.compile(r"^\s*(?:here|sure|certainly|of course|summary\s*:)", re.I)

preamble_counts = Counter()
truncation_counts = Counter()
for ex in per_ex:
    for c in CONDITIONS_RUN:
        payload = ex["conditions"][c]
        raw = payload.get("text_raw", payload["text"])
        if preamble_re.search(raw):
            preamble_counts[c] += 1
        if payload["n_new_tokens"] >= cap - 4:
            truncation_counts[c] += 1

import pandas as pd
diag = pd.DataFrame({
    "preamble_leak (raw)": {c: f"{preamble_counts[c]}/{N}" for c in CONDITIONS_RUN},
    "truncated_at_cap":   {c: f"{truncation_counts[c]}/{N}" for c in CONDITIONS_RUN},
})
print("Preamble leakage and truncation counts (after v2 instruction + strip):")
display(diag)

# Activation rate distribution for the selective conditions.
import matplotlib.pyplot as plt
selective_conds = [c for c in CONDITIONS_RUN if c.startswith("selective")]
fig, ax = plt.subplots(figsize=(7, 3.5))
for c in selective_conds:
    vals = [ex["conditions"][c]["fraction_constrained"] for ex in per_ex
            if ex["conditions"][c]["fraction_constrained"] is not None]
    if not vals:
        continue
    ax.hist(vals, bins=24, alpha=0.45, label=c)
ax.set_xlabel("fraction of decoding steps with constraint active")
ax.set_ylabel("examples")
ax.set_title("Selective detector activation rate (v3: watchdog cap should keep this unimodal)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# List any selective_hard examples still above 0.5 activation (potential outliers).
outliers = sorted(
    [(ex["idx"], ex["conditions"]["selective_hard"]["fraction_constrained"],
      ex["conditions"]["selective_hard"]["text"][:120])
     for ex in per_ex
     if (ex["conditions"]["selective_hard"]["fraction_constrained"] or 0) > 0.5],
    key=lambda t: -t[1],
)
print(f"\n# selective_hard examples with fraction_constrained > 0.5: {len(outliers)}")
for idx, fc, text in outliers[:10]:
    print(f"  idx={idx}  fc={fc:.2f}  {text!r}")


## 7b · Paired statistical tests

In [ ]:
import numpy as np
from scipy.stats import wilcoxon
CONDITIONS_RUN = tuple(result["conditions"])

def paired(metric, cond_a, cond_b):
    diffs, w, t, l = [], 0, 0, 0
    for ex in per_ex:
        a = ex["conditions"][cond_a].get(metric)
        b = ex["conditions"][cond_b].get(metric)
        if a is None or b is None: continue
        d = a - b
        diffs.append(d)
        if d < 0: w += 1
        elif d > 0: l += 1
        else: t += 1
    arr = np.array(diffs)
    # zero_method="wilcox" drops ties (the default); with so many ties this is appropriate.
    try:
        stat, p = wilcoxon(arr, alternative="two-sided", zero_method="wilcox")
    except ValueError:
        stat, p = (float("nan"), 1.0)
    return {
        "mean_diff": float(arr.mean()),
        "median_diff": float(np.median(arr)),
        "wins (a<b)": w, "ties": t, "losses (a>b)": l,
        "wilcoxon_p": float(p),
    }

# Test every non-unconstrained condition against unconstrained.
test_conds = [c for c in CONDITIONS_RUN if c != "unconstrained"]
rows = []
for metric, sign in [("hallucinated_entity_count", "lower is better"),
                     ("rouge1",                    "higher is better")]:
    for cond in test_conds:
        r = paired(metric, cond, "unconstrained")
        rows.append({"metric": metric, "vs": f"{cond} vs unconstrained ({sign})", **r})

stat_df = pd.DataFrame(rows)
print("Paired Wilcoxon vs unconstrained (per-example differences):")
display(stat_df.round(4))


## 7c · Stratified means (excluding runaway selective_hard examples)

In [ ]:
THRESH = 0.4
CONDITIONS_RUN = tuple(result["conditions"])
clean_ex = [ex for ex in per_ex
            if (ex["conditions"]["selective_hard"]["fraction_constrained"] or 0) <= THRESH]
excluded = len(per_ex) - len(clean_ex)
print(f"Stratified on selective_hard fraction_constrained ≤ {THRESH}: "
      f"{len(clean_ex)}/{len(per_ex)} examples kept ({excluded} excluded as runaway/borderline).")

metrics_of_interest = ["entity_precision", "hallucinated_entity_count",
                       "number_accuracy", "rouge1", "rougeL", "length_tokens"]

def cond_mean(condition, metric, examples):
    vals = [ex["conditions"][condition].get(metric) for ex in examples
            if ex["conditions"][condition].get(metric) is not None]
    return float(np.mean(vals)) if vals else float("nan")

rows = []
for cond in CONDITIONS_RUN:
    rows.append({"condition": cond,
                 **{m: cond_mean(cond, m, clean_ex) for m in metrics_of_interest}})

strat = pd.DataFrame(rows).set_index("condition").round(3)
print("\nMeans on the clean subset:")
display(strat)

# Paired Wilcoxon on the clean subset for the headline metrics.
print("\nPaired Wilcoxon on the clean subset (selective_* vs unconstrained):")
def paired_subset(metric, cond_a, cond_b, examples):
    diffs = [ex["conditions"][cond_a][metric] - ex["conditions"][cond_b][metric]
             for ex in examples
             if ex["conditions"][cond_a].get(metric) is not None
             and ex["conditions"][cond_b].get(metric) is not None]
    arr = np.array(diffs)
    try:
        stat, p = wilcoxon(arr, alternative="two-sided", zero_method="wilcox")
    except ValueError:
        stat, p = (float("nan"), 1.0)
    return {"n": len(arr), "mean_diff": float(arr.mean()),
            "median_diff": float(np.median(arr)), "wilcoxon_p": float(p)}

selective_conds = [c for c in CONDITIONS_RUN if c.startswith("selective")]
rows = []
for metric in ["hallucinated_entity_count", "rouge1"]:
    for cond in selective_conds:
        r = paired_subset(metric, cond, "unconstrained", clean_ex)
        rows.append({"metric": metric, "vs": f"{cond} vs unconstrained", **r})
display(pd.DataFrame(rows).round(4))


## 8 · Plot the headline figures

In [ ]:
from scripts.plot_results import plot_all
from pathlib import Path
from IPython.display import Image, display

paths = plot_all(result, Path(FIGURES_DIR))
for path in paths:
    if path.exists():
        display(Image(str(path)))

## 9 · Qualitative samples

In [ ]:
from textwrap import shorten

def precision(example, cond):
    return example["conditions"].get(cond, {}).get("entity_precision", 1.0)

# Examples where selective_hard is more faithful than unconstrained.
candidates = sorted(
    result["per_example"],
    key=lambda ex: precision(ex, "selective_hard") - precision(ex, "unconstrained"),
    reverse=True,
)[:5]

for ex in candidates:
    print("=" * 80)
    print("idx", ex["idx"])
    print("article :", shorten(ex["article"], 200))
    print("reference:", ex["reference"])
    for cond in result["conditions"]:
        payload = ex["conditions"].get(cond, {})
        text = payload.get("text", "")
        p = payload.get("entity_precision")
        h = payload.get("hallucinated_entity_count")
        r = payload.get("rouge1")
        print(f"  [{cond:>23s}] P={p:.2f} hall={h} R1={r:.2f} :: {shorten(text, 160)}")